<a href="https://colab.research.google.com/github/angeshwar-yadav/Evaluating-the-Financial-Fraud-Detection-Model/blob/main/Evaluating_the_Financial_Fraud_Detection_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.1 MB/s eta 0:00:00


In [2]:
!pip install -q transformers peft

In [3]:
from transformers import pipeline

repo_name = "angeshwar/fraud-detector"  # Replace with your repo

finetuned_model = pipeline("text-generation", model=repo_name, device="cuda")
print("Fine-tuned model loaded!")

adapter_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Fine-tuned model loaded!


In [4]:
base_model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device="cuda"
)
print("Base model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded!


In [5]:
!pip install -q datasets

In [6]:
from datasets import load_dataset

dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Dataset size: {len(dataset):,} transactions")

README.md:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): reconstructing file:   0%|          |  0.00B /  138MB            

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): downloading bytes:           |  0.00B            

...

Cifer-Fraud-Detection-Dataset-AF-part-8-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Dataset size: 21,000,000 transactions


In [7]:
from datasets import concatenate_datasets

# Get 50 fraud transactions
fraud_test = dataset.filter(lambda x: x['isFraud'] == 1)
fraud_test = fraud_test.shuffle(seed=99).select(range(50))

# Get 50 legitimate transactions
non_fraud_test = dataset.filter(lambda x: x['isFraud'] == 0)
non_fraud_test = non_fraud_test.shuffle(seed=99).select(range(50))

Filter:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/21000000 [00:00<?, ? examples/s]

In [8]:
# Combine and shuffle
test_data = concatenate_datasets([fraud_test, non_fraud_test])
test_data = test_data.shuffle(seed=99)

print(f"Test set: {len(test_data)} transactions")

Test set: 100 transactions


In [9]:
def create_prompt(row):
    prompt = f"""Analyze this transaction for fraud risk:
- Type: {row['type']}
- Amount: ${row['amount']:,.2f}
- Sender Balance Before: ${row['oldbalanceOrg']:,.2f}
- Sender Balance After: ${row['newbalanceOrig']:,.2f}
- Recipient Balance Before: ${row['oldbalanceDest']:,.2f}
- Recipient Balance After: ${row['newbalanceDest']:,.2f}"""
    return prompt

In [10]:
actual_labels = []

for row in test_data:
    if row['isFraud'] == 1:
        actual_labels.append("HIGH")
    else:
        actual_labels.append("LOW")

print(f"Prepared {len(actual_labels)} labels")

Prepared 100 labels


In [11]:
def get_prediction(model, prompt):
    messages = [{"role": "user", "content": prompt}]
    output = model(messages, max_new_tokens=10, return_full_text=False)[0]
    response = output["generated_text"].upper()
    if "HIGH" in response:
        return "HIGH"
    elif "LOW" in response:
        return "LOW"
    else:
        return "LOW"  # Default if unclear

In [12]:
finetuned_predictions = []
base_predictions = []

for i in range(len(test_data)):
    row = test_data[i]
    prompt = create_prompt(row)

    finetuned_predictions.append(get_prediction(finetuned_model, prompt))
    base_predictions.append(get_prediction(base_model, prompt))

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_data)}...")

print("Done!")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=10) and `max_lengt

Processed 10/100...


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Processed 20/100...


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

...

Processed 80/100...


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Processed 90/100...


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Processed 100/100...
Done!


In [13]:
!pip install -q scikit-learn

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Fine-tuned model
ft_accuracy  = accuracy_score(actual_labels, finetuned_predictions)
ft_precision = precision_score(actual_labels, finetuned_predictions, pos_label="HIGH")
ft_recall    = recall_score(actual_labels, finetuned_predictions, pos_label="HIGH")
ft_f1        = f1_score(actual_labels, finetuned_predictions, pos_label="HIGH")

In [18]:
# Base model
base_accuracy  = accuracy_score(actual_labels, base_predictions)
base_precision = precision_score(actual_labels, base_predictions, pos_label="HIGH")
base_recall    = recall_score(actual_labels, base_predictions, pos_label="HIGH")
base_f1        = f1_score(actual_labels, base_predictions, pos_label="HIGH")

In [19]:
print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)
print(f"Metric       Base Model    Fine-tuned")
print("-" * 50)
print(f"Accuracy     {base_accuracy:.0%}            {ft_accuracy:.0%}")
print(f"Precision    {base_precision:.0%}            {ft_precision:.0%}")
print(f"Recall       {base_recall:.0%}            {ft_recall:.0%}")
print(f"F1 Score     {base_f1:.0%}            {ft_f1:.0%}")
print("=" * 50)

MODEL COMPARISON
Metric       Base Model    Fine-tuned
--------------------------------------------------
Accuracy     48%            52%
Precision    25%            52%
Recall       2%            58%
F1 Score     4%            55%
